# Segmentación de Vasos Retinianos — Colab Runner

Este notebook monta Google Drive, clona/actualiza el repositorio, crea enlaces simbólicos hacia tus datasets en `EP01/pregunta2/datasets`, valida estructura/máscaras y ejecuta el pipeline completo con un único comando: `bash run_all.sh`.

Antes de ejecutar: **Entorno de ejecución → Cambiar tipo de entorno → GPU**.

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Google Drive montado')

## 2. Clonar repositorio, vincular datasets y validar

In [ ]:
REPO_URL = "https://github.com/manadevelop/retinal_segmentation.git"
PROJECT_DIR = "/content/retinal_segmentation"

import os
import sys
import shutil
from pathlib import Path

# Clonar o actualizar repositorio
if not os.path.isdir(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull

%cd {PROJECT_DIR}

# GPU
print("\nVerificando GPU:")
!nvidia-smi

# Rutas reales en Google Drive
BASE = Path("/content/drive/MyDrive/EP01/pregunta2/datasets")
DRIVE_BASE = BASE / "DRIVE"
CHASE_BASE = BASE / "CHASE_DB1/new/chase/test/test"
STARE_BASE = BASE / "STARE"

# DRIVE puede estar en DRIVE/training/... o DRIVE/datasets/training/...
if (DRIVE_BASE / "training" / "images").exists():
    DRIVE_SRC = DRIVE_BASE
elif (DRIVE_BASE / "datasets" / "training" / "images").exists():
    DRIVE_SRC = DRIVE_BASE / "datasets"
else:
    raise FileNotFoundError(
        "No se encontró DRIVE/training/images ni DRIVE/datasets/training/images. "
        f"Ruta revisada: {DRIVE_BASE}"
    )

print("\nRutas detectadas:")
print("DRIVE     :", DRIVE_SRC)
print("CHASE_DB1 :", CHASE_BASE)
print("STARE     :", STARE_BASE)

Path("data").mkdir(exist_ok=True)

def make_link(src, dst, required=True):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.is_symlink() or dst.is_file():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)

    if not src.exists():
        msg = f"No encontrado: {src}"
        if required:
            raise FileNotFoundError(msg)
        print("⚠", msg)
        return False

    os.symlink(str(src), str(dst))
    print(f"✓ {dst} -> {src}")
    return True

# Enlaces a datasets
make_link(DRIVE_SRC, "data/drive", required=True)
make_link(CHASE_BASE / "images", "data/chase_db1/images", required=True)
make_link(CHASE_BASE / "1st_manual", "data/chase_db1/labels", required=True)
make_link(CHASE_BASE / "mask", "data/chase_db1/mask", required=False)
make_link(CHASE_BASE / "2nd_manual", "data/chase_db1/labels-2nd", required=False)
make_link(STARE_BASE / "images", "data/stare/images", required=False)
make_link(STARE_BASE / "masks", "data/stare/masks", required=False)
make_link(STARE_BASE / "masks", "data/stare/labels-ah", required=False)

# Entorno para Python y scripts shell
SRC_DIR = Path(PROJECT_DIR) / "src"
os.environ["DATA_FROM_DRIVE"] = "1"
os.environ["PYTHONPATH"] = f"{SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("\nDATA_FROM_DRIVE:", os.environ["DATA_FROM_DRIVE"])
print("PYTHONPATH:", os.environ["PYTHONPATH"])

print("\nArchivos principales:")
for p in ["requirements.txt", "run_all.sh", "src/data/dataset.py", "src/data/transforms.py", "src/train.py"]:
    print("✓" if Path(p).exists() else "✗", p)

print("\nEstructura montada:")
!find data -maxdepth 3 -type d -o -type l | sort

print("\nValidación de datasets:")
!python scripts/validate_datasets.py --img_size 512


## 3. Ejecutar pipeline completo

In [ ]:
import os
from pathlib import Path

os.chdir('/content/retinal_segmentation')
os.environ['DATA_FROM_DRIVE'] = '1'
os.environ['PYTHONPATH'] = f"{Path.cwd() / 'src'}:{os.environ.get('PYTHONPATH', '')}"

print('Ejecutando pipeline completo...')
print('DATA_FROM_DRIVE=', os.environ['DATA_FROM_DRIVE'])
print('PYTHONPATH=', os.environ['PYTHONPATH'])

!bash run_all.sh

## 4. Opcional: respaldar outputs/results en Google Drive

In [ ]:
import os
import shutil
from pathlib import Path

os.chdir('/content/retinal_segmentation')
backup = Path('/content/drive/MyDrive/EP01/pregunta2/resultados')
backup.mkdir(parents=True, exist_ok=True)

for src in ['outputs', 'results']:
    src_path = Path(src)
    if src_path.exists():
        dst = backup / src
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src_path, dst)
        print(f'✓ {src} -> {dst}')
    else:
        print(f'⚠ No existe {src}; se omite')

## 5. Opcional: resumen de métricas

In [ ]:
import json
from pathlib import Path

os.chdir('/content/retinal_segmentation')
print(f"\n  {'Experimento':<35} {'Sens':>8} {'Spec':>8} {'F1':>8} {'AUC':>8}")
print(f"  {'-'*67}")
for d in sorted(Path('outputs').iterdir()) if Path('outputs').exists() else []:
    f = d / 'test_metrics.json'
    if f.exists():
        m = json.load(open(f))
        print(f"  {d.name:<35} {m.get('sensibilidad', float('nan')):>8.4f} {m.get('especificidad', float('nan')):>8.4f} {m.get('f1', float('nan')):>8.4f} {m.get('auc_roc', float('nan')):>8.4f}")